In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fijamos una semilla para que el azar sea siempre el mismo 
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
df = pd.read_csv('insurance.csv')

# Transformamos categorías en columnas de 0 y 1 ya que nuestro modelo no puede trabajar con texto porque las redes neuronales trabajan con números, no con texto.
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)   #el drop_first=True es para evitar redundacia, es decir que si no es hombre, entonces es mujer, si no es fumador, entonces es no fumador, etc.

# Separamos las características (X) de la variable objetivo (y) que es 'charges'. 
X = df.drop('charges', axis=1).values
y = df['charges'].values.reshape(-1, 1)



In [ ]:
# Separamos 80% para entrenar y 20% para el resto
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Ese 20% lo dividimos en dos: 10% para Validar y 10% para el Test final
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [ ]:


#Estandarizamos los datos porque las redes neuronales funcionan mejor con datos centrados y escalados
#Si no lo hacemos, algunas características podrían dominar a otras debido a sus diferentes escalas, lo que podría dificultar el aprendizaje del modelo.
scaler = StandardScaler()

# Ajustamos el scaler solo con los datos de entrenamiento para evitar "fugas de información" 
X_train = scaler.fit_transform(X_train)

# Luego transformamos los datos de validación y test usando el mismo scaler para mantener la consistencia
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)



# La red neuronal no entiende tablas de Excel o listas normales, solo entiende "Tensores".
# Convertimos los números a Floats (decimales) para que el motor de PyTorch pueda hacer cálculos rápidos.
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val)

X_test, y_test = torch.FloatTensor(X_test), torch.FloatTensor(y_test)

In [ ]:
import torch.nn as nn
import torch.nn.init as init

# Definimos la estructura de nuestra red neuronal. 
class RedSeguros(nn.Module):
    def __init__(self, input_dim):
        super(RedSeguros, self).__init__()
        
        # CAPA 1: Recibe los datos y los pasa a 64 neuronas
        self.capa1 = nn.Linear(input_dim, 64)
        
        # CAPA 2: Pasa de 64 neuronas a 32 neuronal ya que hacer redes profundas ("Deep") es más eficiente que redes anchas.
        self.capa2 = nn.Linear(64, 32)
        
        # CAPA DE SALIDA: De 32 neuronas a 1 sola que es el valor que queremos predecir (el costo del seguro).
        self.salida = nn.Linear(32, 1)
        
        # FUNCIÓN DE ACTIVACIÓN: Sirve para que la red aprenda relaciones complejas, no solo líneas rectas.
        self.relu = nn.ReLU()
        
        # INICIALIZACIÓN DE HE/KAIMING NORMAL: Esto ayuda a que la red aprenda mejor al empezar con pesos adecuados.
        init.kaiming_normal_(self.capa1.weight)
        init.kaiming_normal_(self.capa2.weight)
        init.kaiming_normal_(self.salida.weight)

    def forward(self, x):
        # Aquí definimos el camino que siguen los datos
        x = self.relu(self.capa1(x)) # Pasa por capa 1 y aplica ReLU
        x = self.relu(self.capa2(x)) # Pasa por capa 2 y aplica ReLU
        x = self.salida(x)           # Pasa por la salida (aquí no hay ReLU porque es regresión)
        return x

# Creamos el modelo usando el número de columnas que tiene X_train
modelo = RedSeguros(X_train.shape[1])





In [ ]:
#necesitamos ahora ver cuanto se equivoca la red y como mejorarlo

# La función de pérdida (loss function) mide cuánto se equivoca el modelo. 
# Para problemas de regresión como este, es común usar el MSE que penaliza más los errores grandes.
criterion = nn.MSELoss()

# El optimizer va a ser  el algoritmo que ajusta los pesos de la red para minimizar la pérdida.
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)   ## Usamos Adam porque es más rápido que el Gradient Descent normal y se adapta solo.

In [ ]:
#Este es el bucle de entrenamiento 

# Decidimos cuántas veces va a estudiar el dataset completo
epochs = 300 
historial_error = []

for epoch in range(epochs):
    # Ponemos el modelo en modo entrenamiento
    modelo.train()
    
    # Reseteamos los gradientes antes de cada paso para que no se acumulen
    optimizer.zero_grad()
    
    # La red hace su predicción con los datos de entrenamiento
    predicciones = modelo(X_train)
    
    # Comparamos la predicción con el precio real
    loss = criterion(predicciones, y_train)
    
    #  La red va hacia atrás para ver quién tuvo la culpa del error
    loss.backward()
    
    # El optimizador ajusta los pesos basándose en el error
    optimizer.step()
    
    # Guardamos el error para ver luego si ha bajado
    historial_error.append(loss.item())
    
    # Imprimimos el progreso cada 50 pasos
    if (epoch + 1) % 50 == 0:
        print(f'Época {epoch+1}/{epochs} - Error actual: {loss.item():.4f}')

In [ ]:
#convertir el TEST a tensores 
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

#creamos una lista para guardar el error de validación
historial_v_error = []